# N096 · 堆贪心与撤销局部决策

**目标：** 在资源受限时替换已选集合中的最差元素。

**先修：** N057, N061, N093。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 截止期排序；最大堆替换；可用项目；预算阈值；交换证明。

## 从问题到算法

堆贪心的作用是允许撤销先前决定。按截止时间处理课程，超时就删除已选课程中最长的一门：数量相同而总时长更小，留给以后更多空间。IPO则不同：已满足资本门槛的项目中选利润最大者，但这里利润是净增加，不扣除门槛资本。

## 最小实现

**本章接口：** `schedule_course(courses)`、`find_maximized_capital(k, w, profits, capital)`

In [1]:
from heapq import heappush,heappop

def schedule_course(courses):
    total=0; chosen=[]
    for duration,deadline in sorted(courses,key=lambda c:c[1]):
        heappush(chosen,-duration); total+=duration
        if total>deadline: total+=heappop(chosen)
    return len(chosen)

def find_maximized_capital(k,w,profits,capital):
    projects=sorted(zip(capital,profits)); i=0; available=[]
    for _ in range(k):
        while i<len(projects) and projects[i][0]<=w:
            heappush(available,-projects[i][1]); i+=1
        if not available or available[0]>=0: break
        w-=heappop(available)
    return w

## 正确性、前提与复杂度

课程按截止期交换排序不损害可行性。加入新课程若超时，必须至少放弃一门，删除最长者在保持最多数量的候选中使占用时间最小。IPO利润非负时，先选更大利润不会减少后续可选项目集合，可将最优顺序首项替换为当前最大收益项目。

**代价：** O(n log n+k log n)时间上界，O(n)辅助空间。IPO假定最多选k项、每项只能一次、capital是启动门槛；负收益并非本章标准题域，实现会提前停在非正最大收益处。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

courses=[[100,200],[200,1300],[1000,1250],[2000,3200]]; heap=[]; total=0; rows=[]
for duration,deadline in sorted(courses,key=lambda x:x[1]):
    heappush(heap,-duration); total+=duration; removed=None
    if total>deadline: removed=-heappop(heap); total-=removed
    rows.append((duration,deadline,removed,total,len(heap)))
show_table(['时长','截止','撤销','累计时长','课程数'],rows)

时长,截止,撤销,累计时长,课程数
100,200,None,100,1
1000,1250,None,1100,2
200,1300,None,1300,3
2000,3200,2000,1300,3


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert schedule_course([[100,200],[200,1300],[1000,1250],[2000,3200]])==3
assert schedule_course([[5,4]])==0
assert find_maximized_capital(2,0,[1,2,3],[0,1,1])==4
assert find_maximized_capital(3,0,[5],[1])==0
from itertools import combinations
from random import Random
rng=Random(96)
for _ in range(100):
    a=[(rng.randrange(1,7),rng.randrange(1,15)) for _ in range(rng.randrange(8))]; best=0
    for k in range(len(a)+1):
        for group in combinations(a,k):
            total=0; valid=True
            for t,d in sorted(group,key=lambda x:x[1]):
                total+=t
                if total>d: valid=False
            if valid: best=max(best,k)
    assert schedule_course(a)==best

def brute(k,w,p,c,used=0):
    if not k: return w
    return max([w]+[brute(k-1,w+p[i],p,c,used|1<<i) for i in range(len(p)) if not used>>i&1 and c[i]<=w])
for _ in range(60):
    p=[rng.randrange(6) for _ in range(5)]; c=[rng.randrange(8) for _ in range(5)]
    assert find_maximized_capital(3,1,p,c)==brute(3,1,p,c)
print('N096: 所有本章断言通过')

N096: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 构造永不撤销选择导致次优的实例。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 用子集穷举作参照。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 630. Course Schedule III（canonical）
- 502. IPO（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。